# 04. Fine-Tuning y Comparativa de Fashion-CLIP vs Adaptador Estético

Este notebook implementa el pipeline completo de Fine-Tuning de compatibilidad estética (128-D).

### Estructura en Google Drive (`Mi unidad / MACHINE-BULLS`):
```
Mi unidad / MACHINE-BULLS/
├── docs/
├── sample_data/               # 12,000 pares de entrenamiento y evaluación
├── models/                    # Aquí se guarda fashion_clip_adapter.pt
├── 01_baseline_stylesync_garments2look.ipynb
├── 03_clip_zero_shot.ipynb
└── 04_clip_fine_tuning.ipynb
```


In [ ]:
import platform
import sys
from pathlib import Path

# Detección automática de entorno (Google Colab vs Local)
IS_COLAB = Path("/content").exists()

if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("[OK] Google Drive montado exitosamente en /content/drive")
    except Exception as exc:
        print(f"[WARN] No se pudo montar Google Drive: {exc}")

    # Ruta canónica en Drive: Mi unidad / MACHINE-BULLS
    DRIVE_PROJECT = Path("/content/drive/MyDrive/MACHINE-BULLS")
    if DRIVE_PROJECT.exists():
        REPO_ROOT = DRIVE_PROJECT
    elif Path("/content/MACHINE-BULLS").exists():
        REPO_ROOT = Path("/content/MACHINE-BULLS")
    else:
        REPO_ROOT = Path("/content")
    print(f"[OK] Entorno Google Colab detectado | Proyecto: {REPO_ROOT}")
else:
    REPO_ROOT = Path.cwd().resolve()
    for parent in [REPO_ROOT, *REPO_ROOT.parents]:
        if (parent / "sample_data").exists() or (parent / ".git").exists():
            REPO_ROOT = parent
            break
    print(f"[OK] Entorno Local detectado | Repo: {REPO_ROOT}")

SAMPLE_DIR = REPO_ROOT / "sample_data"
DOCS_DIR = REPO_ROOT / "docs"
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"[RUTAS] sample_data: {SAMPLE_DIR} (existe: {SAMPLE_DIR.exists()})")
print(f"[RUTAS] docs: {DOCS_DIR} (existe: {DOCS_DIR.exists()})")
print(f"[RUTAS] models: {MODELS_DIR}")
print(f"Python: {sys.version.split()[0]} | Plataforma: {platform.platform()}")


In [ ]:
# Instalación de dependencias en Google Colab
if IS_COLAB:
    !pip install -q transformers dagshub mlflow pyyaml pillow
    print("[OK] Dependencias instaladas en Google Colab")


In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Cargar variables de entorno si existen en Drive o en el repo
for env_candidate in [REPO_ROOT / ".env", REPO_ROOT / "backend" / ".env", Path("/content/MACHINE-BULLS/.env")]:
    if env_candidate.exists():
        load_dotenv(env_candidate)

token = os.environ.get("DAGSHUB_USER_TOKEN") or os.environ.get("DAGSHUB_TOKEN")
if token:
    os.environ["DAGSHUB_TOKEN"] = token

import dagshub
import mlflow

try:
    dagshub.init(repo_owner="jesusrodrigov", repo_name="machine-bulls", mlflow=True)
    print("[OK] Conectado exitosamente a DagsHub MLflow")
except Exception as ex:
    print(f"[INFO] MLflow tracking local: {ex}")


In [ ]:
# Asegurar que el modulo src de backend este en el path
# Resuelve si el backend vive en Drive o en el clon de /content/MACHINE-BULLS
candidate_paths = [
    REPO_ROOT / "backend" / "src",
    Path("/content/MACHINE-BULLS/backend/src"),
    Path("/content/backend/src"),
    REPO_ROOT / "src",
]
for p in candidate_paths:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import torch
from ml.augmentation import verificar_splits_anti_fuga, get_image_transforms
from ml.models.adapter import AestheticProjectionHead
from ml.models.prompts import ESTETICAS_AMPLIADAS
from ml.training.train import train, load_config
from ml.validation.evaluate_fitb import evaluate_fitb
from ml.validation.temperature_scaling import ModelWithTemperature
from ml.validation.run_comparison import run_baseline_evaluation, run_adapter_evaluation, print_comparison_table

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[OK] Módulos de StyleSync importados correctamente. Dispositivo activo: {device}")
print(f"[INFO] Catálogo ampliado de estéticas ({len(ESTETICAS_AMPLIADAS)}): {list(ESTETICAS_AMPLIADAS)}")


In [ ]:
# Configurar y ejecutar entrenamiento del adaptador (D=512 -> D=128)
config_path = REPO_ROOT / "backend" / "src" / "ml" / "config.yaml"
if not config_path.exists():
    config_path = Path("/content/MACHINE-BULLS/backend/src/ml/config.yaml")

cfg = load_config(config_path) if config_path.exists() else {
    "model": "patrickjohncyh/fashion-clip",
    "dataset": {"name": "sample_data_clip_pairs"},
    "training": {"epochs": 5 if IS_COLAB else 1, "batch_size": 32 if IS_COLAB else 8, "lr": 0.001},
    "seed": 42
}

print(">>> Iniciando entrenamiento del Adaptador Estético sobre sample_data/...")
train_results = train(cfg, fast_dev_run=False, use_mlflow=True, run_name="colab-fashion-clip-adapter")
print("[OK] Entrenamiento completado:", train_results)


In [ ]:
# Guardar y respaldar checkpoint en Google Drive: Mi unidad / MACHINE-BULLS / models
import shutil

local_ckpt = Path(train_results["checkpoint_path"])
drive_folder = Path("/content/drive/MyDrive/MACHINE-BULLS/models")
drive_folder.mkdir(parents=True, exist_ok=True)
drive_dest = drive_folder / "fashion_clip_adapter.pt"

if IS_COLAB and Path("/content/drive/MyDrive").exists():
    shutil.copy(local_ckpt, drive_dest)
    print(f"[EXITO] Checkpoint copiado a Google Drive en: {drive_dest}")
    print("[RUTA DIRECTA] Mi unidad / MACHINE-BULLS / models / fashion_clip_adapter.pt")
    print(">>> Descárgalo a tu equipo y colócalo en 'models/fashion_clip_adapter.pt' de tu proyecto local.")
else:
    print(f"[INFO] Checkpoint local disponible en: {local_ckpt}")


In [ ]:
# Ejecutar comparativa formal completa (Baseline Zero-Shot vs Adaptador 128-D)
b_metrics = run_baseline_evaluation()
a_metrics = run_adapter_evaluation()
print_comparison_table(b_metrics, a_metrics)
